# Strict-AOI cell mask

zagg's optional `output.aoi_mask` (default off) packages a per-cell boolean
aligned to the output cell grid — `True` where the cell falls inside the area of
interest (AOI). It is **"package, don't clip"**: no observation is dropped, and a
flag-off run is byte-identical to today. See the narrative doc `docs/aoi_mask.md`
for the config and storage layout.

This notebook is fully self-contained — it needs no remote data — and runs
anywhere `zagg` (with `mortie>=0.8.2`) is installed. Binder launch additionally
requires the repo-wide `.binder/` environment, which lands separately via #105; a
Binder badge is omitted here until that infrastructure exists, since a default
Binder build can't resolve `zagg` + `mortie>=0.8.2`.

It builds a small HEALPix grid and an AOI box, computes the mask the same way the
shard-map build stage does (native morton, no lat/lon decode), and shows the mask
is `True` exactly for the in-AOI cells (issue #101).

In [ ]:
import numpy as np
from zagg.grids import HealpixGrid

# A small HEALPix grid. parent_order is the shard order; child_order the leaf
# (cell) order the mask resolves at.
grid = HealpixGrid(parent_order=4, child_order=8, layout="fullsphere")

# An AOI box in WGS84, as the [(lats, lons), ...] parts list `coverage` takes.
lat0, lon0, lat1, lon1 = 10.0, 10.0, 20.0, 20.0
lats = np.array([lat0, lat0, lat1, lat1, lat0])
lons = np.array([lon0, lon1, lon1, lon0, lon0])
parts = [(lats, lons)]

# 1) Compact MOC of the AOI at child_order (built once at the shard-map stage).
aoi_moc = grid.aoi_moc(parts)
print(f"AOI MOC: {aoi_moc.size} mixed-order cells")

In [ ]:
# 2) Pick a shard the AOI touches and expand its per-shard sub-MOC to a per-cell
#    boolean over the shard's children() — already in cell/storage order.
from mortie import moc_to_order

flat = np.unique(np.asarray(moc_to_order(aoi_moc, grid.child_order), dtype=np.uint64))
shard_key = int(grid.shards_of(grid.cells_of(flat[:1]))[0])
children = grid.children(shard_key)

shard_moc = grid.aoi_shard_moc(aoi_moc, shard_key)
mask = grid.aoi_mask_for_children(shard_moc, children)

print(f"shard {shard_key}: {len(children)} cells, {int(mask.sum())} in-AOI")

# The mask equals: which of this shard's children land in the AOI cover.
expected = np.isin(np.asarray(children, dtype=np.uint64), flat)
assert np.array_equal(mask, expected)
print("mask matches the cell-order AOI cover ✓")

In [ ]:
# 3) Visualize the in-AOI cells by their HEALPix center (decode only for the
#    picture — the mask itself never decodes centers).
import matplotlib.pyplot as plt
from mortie.tools import mort2geo

clat, clon = mort2geo(np.asarray(children))
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(clon[~mask], clat[~mask], s=8, c="lightgray", label="out of AOI")
ax.scatter(clon[mask], clat[mask], s=8, c="C0", label="in AOI")
ax.add_patch(plt.Rectangle((lon0, lat0), lon1 - lon0, lat1 - lat0,
                           fill=False, edgecolor="red", lw=1.5, label="AOI"))
ax.set_xlabel("lon"); ax.set_ylabel("lat"); ax.legend(loc="upper left")
ax.set_title(f"shard {shard_key}: strict-AOI cell mask")
plt.show()

Once a run with `output.aoi_mask: true` has written the store, filtering to the
strict AOI is just a selection on the `aoi_mask` array:

```python
import xarray as xr
ds = xr.open_zarr("s3://bucket/atl06.zarr/13")  # child_order=13 group
strict = ds.where(ds["aoi_mask"])               # NaN out the overhang cells
```